# 11 Från notebook till pipeline

Notebooks är bra för att utforska och förklara. Vi kör en cell i taget, tittar på resultatet och skriver ner vad vi lär oss. Men när vi vet vad som ska göras och vill göra det igen, varje vecka eller varje månad, blir notebooks opraktiska. Någon måste öppna dem, köra dem i rätt ordning och hålla koll på att inget glömts.

En pipeline är samma steg samlade i vanliga Python-filer som körs från början till slut med ett enda kommando. Den kan köras utan att någon tittar på, till exempel på en server varje natt.

Vi har flyttat koden från notebook 03 till 09 till mappen `pipeline`. Analysen i notebook 10 ligger kvar som notebook, eftersom den handlar om att titta och tolka.

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

PIPELINE = Path("..") / "pipeline"

sorted(fil.name for fil in PIPELINE.glob("*.py"))

## Ett steg per fil

Varje fil motsvarar ett steg i arbetet.

| Fil | Gör | Motsvarar |
| --- | --- | --- |
| `hamta.py` | Hämtar från Webperf, Wikidata, CrUX och Kolada och sparar rådatan | notebook 03, 04, 05 och början av 07 |
| `tvatta.py` | Gör om rådatan till en tvättad tabell per källa | notebook 06 |
| `harmonisera.py` | Bygger gemensamma nycklar och slår ihop tabellerna | notebook 07 |
| `validera.py` | Prövar tabellen mot reglerna | notebook 08 |
| `lagra.py` | Sparar tabellen som Parquet, CSV och i DuckDB | notebook 09 |
| `gemensamt.py` | Sökvägar och funktioner som flera steg använder | |
| `kor.py` | Kör stegen i ordning | |

En skillnad mot notebooks är att all hämtning nu ligger i ett och samma steg. I notebooks hämtade vi Kolada först i notebook 07. I pipelinen hämtar `hamta.py` från alla källor, och `tvatta.py` tvättar alla. Det gör det lättare att köra om tvättningen utan att hämta något.

## Skriv en gång, använd överallt

Funktionen som gör om en adress till ett origin skrev vi i både notebook 05 och 06. I pipelinen finns den en gång, i `gemensamt.py`, och används av de steg som behöver den. Rättar vi ett fel i den rättas det överallt.

Vi kan använda funktionerna i en notebook också. `sys.path` är listan med mappar där Python letar efter moduler att importera. Vi lägger till mappen `pipeline` först i listan.

In [ ]:
sys.path.insert(0, str(PIPELINE.resolve()))

from gemensamt import till_origin

till_origin("https://www.hoganas.se/Invanare/")

## Filen som kör allt

`kor.py` är kort. Den anropar stegen i tur och ordning och skriver ut vad den gör. Vi läser filen och skriver ut den.

In [ ]:
print((PIPELINE / "kor.py").read_text(encoding="utf-8"))

Raden `if __name__ == "__main__":` betyder att koden under den bara körs när filen startas som ett program, och inte när den importeras av en annan fil. `argparse` läser inställningar från kommandoraden. Här finns en, `--utan-hamtning`, som hoppar över hämtningen och använder rådatan som redan finns.

Validering kommer före lagring. Om någon regel inte håller avbryts pipelinen innan den skriver över den data som redan finns sparad. Då ligger det senaste godkända resultatet kvar.

## Köra pipelinen

I en terminal, i mappen `python`, kör vi hela kedjan med ett kommando.

```sh
uv run python pipeline/kor.py
```

Det tar några minuter, mest för att vi väntar mellan frågorna till CrUX. Med `--help` ser vi vilka inställningar som finns.

Vi kan också starta pipelinen härifrån. `subprocess.run` startar ett program, som om vi skrivit kommandot i en terminal. `sys.executable` är den Python som kör den här notebooken, alltså projektets. För att inte anropa källorna igen kör vi med `--utan-hamtning`.

Först sparar vi den tabell som notebook 07 tog fram, så att vi kan jämföra med pipelinens resultat. Har du kört notebooks i ordning är det den filen som ligger i `data/harmoniserad`.

In [ ]:
fil = Path("..") / "data" / "harmoniserad" / "webbplatser.parquet"
fran_notebooks = pd.read_parquet(fil)

resultat = subprocess.run(
    [sys.executable, str(PIPELINE / "kor.py"), "--utan-hamtning"],
    capture_output=True, text=True, check=True,
)
print(resultat.stdout)

`capture_output=True` och `text=True` gör att vi kan skriva ut det programmet skrev. `check=True` gör att cellen stoppar med ett fel om pipelinen misslyckas.

## Samma resultat?

En pipeline ska göra exakt samma sak som de notebooks den ersätter. `pd.testing.assert_frame_equal` jämför två tabeller, med värden, kolumnnamn och datatyper. Om något skiljer stoppar den med ett fel som visar var. Om inget händer är tabellerna identiska.

In [ ]:
fran_pipeline = pd.read_parquet(fil)

pd.testing.assert_frame_equal(fran_notebooks, fran_pipeline)
print("Tabellerna är identiska,", fran_pipeline.shape)

## Köra regelbundet

En pipeline som kan köras med ett kommando kan också schemaläggas. På macOS och Linux finns verktyget cron. En rad som denna i schemat, som öppnas med `crontab -e`, kör pipelinen klockan sex varje måndag och sparar utskriften i en loggfil.

```
0 6 * * 1 cd /sökväg/till/python && uv run python pipeline/kor.py >> data/pipeline.log 2>&1
```

De fem första fälten är minut, timme, dag i månaden, månad och veckodag. En stjärna betyder alla. Veckodag 1 är måndag. `>>` lägger till utskriften sist i filen och `2>&1` tar med felmeddelanden.

Varje körning skriver över rådatan med nya svar. Vill vi kunna jämföra över tid behöver vi spara varje körning för sig, till exempel i en mapp per datum. Det är ett naturligt nästa steg.

## Det här vet vi nu

- Notebooks passar för att utforska och förklara, en pipeline för att köra samma steg igen.
- En pipeline delar upp arbetet i steg, ett per fil, och kör dem med ett kommando.
- Kod som flera steg behöver skrivs en gång och importeras.
- Validering före lagring gör att data som inte håller aldrig skriver över data som gör det.
- En pipeline ska ge exakt samma resultat som de notebooks den ersätter, och det kan vi kontrollera.
- Det som går att köra med ett kommando går också att schemalägga.

Det här var sista steget i serien. Vi har gått från ett första anrop till ett API till en kedja som hämtar, tvättar, harmoniserar, validerar och lagrar data från fyra öppna källor, och en analys som korsrefererar dem.